# Module 6.2: Automated Encrypted 5 TB Google Drive Cloud Backups (`rclone crypt`), Tailscale Subnet Routing & Full-Cluster Observability

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 6: Automated 3-2-1 Cloud Backups & Observability

## 🎯 Learning Goals

By completing this final Chapter 3 capstone laboratory module, you will be able to:

1. Leverage the **5 TB Google Drive storage included with your Gemini Pro (Google AI Pro) subscription** as a **$0 additional cost** offsite backup tier that covers 100% of your **~3.64 TiB RAID-1 `UNAS 2`** capacity—completing the **3-2-1 backup standard**.
2. Compare **UniFi Drive Native Google Drive Cloud Backup** vs. **`rclone crypt` Zero-Knowledge Client-Side Encryption** (NaCl SecretBox / XSalsa20 + Poly1305) and understand why encrypting sensitive emails, tax documents, and private photos *before* they leave your LAN maximizes privacy.
3. Engineer an automated daily **`systemd` cloud backup timer** on `homelab-gpu` (`04:30 AM`) that gracefully respects **Google Drive's strict 750 GB/day upload cap** (`--max-transfer 700G --drive-stop-on-upload-limit`) and safely exports your `rclone.conf` recovery keys offline.
4. Configure a **Tailscale Subnet Router** on `homelab-pi` to securely access `UNAS 2` (`10.0.0.202`) and `USW-Enterprise-8-PoE` (`10.0.0.203`) remotely with zero open ports.
5. Expand **Uptime Kuma (`10.0.0.200:3001`)** to monitor all **10 homelab infrastructure & application pillars** across your 3-node cluster.

## 6.2.1 Conceptual Background: Turning Your Included 5 TB Gemini Pro Storage Into a Zero-Knowledge Vault

Look at the capacity math of your homelab—it aligns almost miraculously:
- **Local Redundant Tier (`UNAS 2`, 2× 4 TB RAID 1):** **`~3.64 TiB`** max usable capacity.
- **Included Cloud Tier (Gemini Pro / Google AI Pro 5 TB Plan):** **`5.0 TB`** of Google Drive storage you *already* pay for!
- Even if you completely fill your `UNAS 2` to 100% capacity, your entire homelab—Immich photos & DB dumps, migrated University OneDrive archive, Paperless-ngx OCR vault, Mail-Archiver `.edu`/Gmail database, Pi-hole, Uptime Kuma, and UniFi Network backups—fits comfortably inside your **5 TB Google Drive** with **>1.3 TB of headroom to spare** for versioning and personal cloud files!

### 1. Native UniFi Drive Cloud Backup vs. `rclone crypt` (Which Should You Use?)

| Feature | Option A: **`rclone crypt` on `homelab-gpu` (Recommended Primary)** | Option B: **UniFi Drive Built-In Google Drive Backup** |
| :--- | :--- | :--- |
| **Zero-Knowledge Privacy** | **Encrypts every filename, folder name, and file content client-side** using NaCl SecretBox (XSalsa20 + Poly1305) *before* upload. Even if someone accesses your Google Drive, they see only random ciphertext blobs! | Uploads directly from `UNAS 2` via UniFi Drive UI; convenient one-click GUI, though less customizable for per-folder retention and Google's 750 GB daily limit. |
| **Google 750 GB/Day Upload Cap Handling** | Natively throttled/capped via **`--max-transfer 700G --drive-stop-on-upload-limit`** so an initial multi-terabyte upload seamlessly resumes across 2–4 nights without `403 Rate Limit Exceeded` lockouts! | Less granular control over daily transfer caps and bandwidth schedules. |
| **Disaster Recovery Portability** | Decrypt and restore your entire homelab onto *any* Linux, macOS, or Windows computer anywhere on Earth using just free `rclone` + your saved encryption passwords—zero proprietary NAS hardware required! | Easy restore inside UniFi Drive, or manual download from Google Drive. |

> 💡 **Pro Tip:** You can use **`rclone crypt`** (configured below!) for your **encrypted 3-2-1 disaster recovery vault** (`gdrive-crypt:`), and optionally use Google Drive normally for day-to-day collaborative Docs/Sheets—keeping your private homelab archives completely opaque in a single tidy `Homelab_Encrypted_Vault/` folder!

### 2. Understanding Google Drive's `750 GB / 24h` Upload Limit
Google Drive enforces a hard **750 GB per 24-hour rolling upload cap** per account. If you try to push 1.5 TB on Day 1 without a guard, Google locks uploads with HTTP `403 userRateLimitExceeded` for 24 hours. By adding:
- **`--max-transfer 700G`** (stops cleanly at 700 GB/night so subsequent nights pick up right where it left off)
- **`--drive-stop-on-upload-limit`** (clean immediate exit if Google's quota is ever reached)
- **`--bwlimit 08:00,5M 23:00,off`** (optional daytime throttle so uploads never saturate your home ISP uplink during video calls, running full speed overnight!)

your initial seed finishes automatically over a few nights, and every nightly incremental backup after that transfers only a few hundred megabytes in seconds!

## 6.2.2 Step-by-Step Laboratory Guide

### Task 6.2.1: Configure `gdrive` + Zero-Knowledge `gdrive-crypt` Remotes in `rclone`
1. On your **personal workstation** (with a browser), authorize `rclone` against the Google account that holds your **5 TB Gemini Pro / Google One** subscription:
   ```bash
   rclone authorize "drive"
   ```
   Sign into your Google account, grant access, and copy the resulting `{"access_token":...}` JSON block.
2. SSH into **`homelab-gpu`** (`ssh homelab-gpu`) and run `rclone config` to create the underlying `gdrive` remote:
   - Press `n` (New remote) $\rightarrow$ Name: **`gdrive`**
   - Storage type: **`drive`** (Google Drive)
   - `client_id` / `client_secret`: Press Enter to leave blank *(or optionally create your own free Google Cloud OAuth Desktop Client ID for dedicated API quota!)*
   - `scope`: **`1`** (Full access to all files)
   - `service_account_file`: Press Enter (blank)
   - `Edit advanced config?`: **`n`**
   - `Use web browser to automatically authenticate?`: **`n`**
   - Paste the `{"access_token":...}` JSON token from Step 1!
   - `Configure as a Shared Drive?`: **`n`** $\rightarrow$ Confirm **`y`**.
3. Still inside `rclone config`, wrap `gdrive` in a **zero-knowledge encrypted `crypt` overlay** named **`gdrive-crypt`**:
   - Press `n` (New remote) $\rightarrow$ Name: **`gdrive-crypt`**
   - Storage type: **`crypt`** (Encrypt/Decrypt a remote)
   - Remote to encrypt/decrypt: **`gdrive:Homelab_Encrypted_Vault`**
   - Filename encryption: **`1`** (`standard` — encrypts all filenames!)
   - Directory name encryption: **`1`** (`true` — encrypts all folder names!)
   - Password: Press **`g`** to generate a strong **256-bit** encryption password!
   - Salt (`password2`): Press **`g`** to generate a second **256-bit** salt password!
   - Confirm **`y`** $\rightarrow$ press `q` to quit `rclone config`.
4. **CRITICAL DISASTER-RECOVERY STEP — Save Your `rclone.conf` Keys Offline!**
   If your house burns down and you need to decrypt `Homelab_Encrypted_Vault` from Google Drive, you **MUST** have both encryption passwords (or a copy of `~/.config/rclone/rclone.conf`) stored securely in your **password manager (Bitwarden / 1Password / Apple Passwords)** or printed on an emergency recovery sheet!
   ```bash
   # View your config to copy the [gdrive-crypt] password & password2 into your password manager:
   cat ~/.config/rclone/rclone.conf
   chmod 600 ~/.config/rclone/rclone.conf
   ```
5. Test encrypted round-trip verification right now:
   ```bash
   echo "Zero-knowledge 3-2-1 test $(date)" > /tmp/crypt_test.txt
   rclone copy /tmp/crypt_test.txt gdrive-crypt:test/
   echo "--- What Google Sees (Encrypted Ciphertext Filenames): ---"
   rclone ls gdrive:Homelab_Encrypted_Vault/
   echo "--- What You See Locally via gdrive-crypt: ---"
   rclone cat gdrive-crypt:test/crypt_test.txt
   # Clean up test file
   rclone purge gdrive-crypt:test/ && rm -f /tmp/crypt_test.txt
   ```
   Look at those two outputs—Google Drive stores completely unreadable scrambled filenames and AES/NaCl ciphertext, while `gdrive-crypt:` transparently decrypts your exact plaintext!

### Task 6.2.2: Automate Nightly Encrypted Offsite Backups (`04:30 AM` `systemd` Timer on `homelab-gpu`)
We schedule the cloud sync at **04:30 AM**—comfortably *after* `homelab-pi`'s `03:00 AM` backup and `homelab-gpu`'s `03:30 AM` database dumps have finished landing on `UNAS 2`!

1. Create `/usr/local/bin/rclone-gdrive-backup.sh` on `homelab-gpu`:
   ```bash
   sudo tee /usr/local/bin/rclone-gdrive-backup.sh > /dev/null << 'EOF'
   #!/usr/bin/env bash
   set -euo pipefail

   LOG_FILE="/var/log/rclone-gdrive-backup.log"
   RCLONE_CFG="/home/pi/.config/rclone/rclone.conf"

   echo "=== [$(date)] Starting encrypted 3-2-1 cloud backup to 5TB Google Drive ===" | tee -a "$LOG_FILE"

   # 1. Trigger and verify both UNAS 2 NFS automounts before syncing!
   ls /mnt/unas_backups /mnt/unas_media >/dev/null 2>&1 || true
   if ! mountpoint -q /mnt/unas_backups || ! mountpoint -q /mnt/unas_media; then
       echo "ERROR: UNAS 2 NFS mounts unavailable! Aborting cloud backup." | tee -a "$LOG_FILE"
       exit 1
   fi

   # Common safe flags respecting Google Drive's 750 GB/day upload ceiling
   COMMON_FLAGS=(
       --config "$RCLONE_CFG"
       --max-transfer 700G
       --drive-stop-on-upload-limit
       --fast-list
       --transfers 4
       --checkers 8
       --stats 30s
       --log-file "$LOG_FILE"
       --log-level INFO
   )

   # 2. Sync HomelabBackups (Immich DB + Photos, Paperless DB, Mail-Archiver DB, UniFi, Pi-hole, Kuma)
   #    Uses --backup-dir so deleted/overwritten files stay recoverable in a dated attic folder for 30 days!
   ATTIC_DATE=$(date +%F)
   /usr/bin/rclone sync /mnt/unas_backups "gdrive-crypt:HomelabBackups" \
       --backup-dir "gdrive-crypt:_Attic/HomelabBackups/${ATTIC_DATE}" \
       "${COMMON_FLAGS[@]}"

   # 3. Copy MediaArchive (University OneDrive Archive, Paperless PDFs, Personal Files)
   #    Uses 'copy' (append-only) so accidental local deletion NEVER deletes your cloud archive!
   /usr/bin/rclone copy /mnt/unas_media "gdrive-crypt:MediaArchive" \
       "${COMMON_FLAGS[@]}"

   echo "=== [$(date)] Encrypted cloud backup finished cleanly! ===" | tee -a "$LOG_FILE"
   EOF
   sudo chmod +x /usr/local/bin/rclone-gdrive-backup.sh
   sudo touch /var/log/rclone-gdrive-backup.log
   ```
   *(Engineering highlight: Notice `HomelabBackups` uses `--backup-dir` attic versioning, while `MediaArchive` uses append-only `rclone copy`—giving you ransomware and fat-finger protection even in the cloud!)*

2. Create and enable the daily **04:30 AM** `systemd` service and timer:
   ```bash
   sudo tee /etc/systemd/system/rclone-gdrive-backup.service > /dev/null << 'EOF'
   [Unit]
   Description=Nightly Encrypted rclone crypt Backup from UNAS 2 to 5TB Google Drive
   After=network-online.target
   Wants=network-online.target

   [Service]
   Type=oneshot
   ExecStart=/usr/local/bin/rclone-gdrive-backup.sh
   EOF

   sudo tee /etc/systemd/system/rclone-gdrive-backup.timer > /dev/null << 'EOF'
   [Unit]
   Description=Run Encrypted Google Drive Cloud Backup Daily at 04:30 AM

   [Timer]
   OnCalendar=*-*-* 04:30:00
   Persistent=true

   [Install]
   WantedBy=timers.target
   EOF

   sudo systemctl daemon-reload
   sudo systemctl enable --now rclone-gdrive-backup.timer
   sudo systemctl list-timers --all | grep -E "rclone|homelab"
   ```
3. **Trigger your initial dry-run check (`--dry-run`) or live seed run now:**
   ```bash
   # Preview what will upload with zero changes first:
   rclone copy /mnt/unas_backups gdrive-crypt:HomelabBackups --dry-run --stats 5s

   # When satisfied, trigger the real service (runs safely in background via systemd):
   sudo systemctl start rclone-gdrive-backup.service --no-block
   tail -f /var/log/rclone-gdrive-backup.log
   ```

### 🖼️ Task 6.2.2 Verification: Zero-Knowledge Encrypted Google Drive Backup & Timer Active

**Verification Objective:** Capture a screenshot showing your `rclone-gdrive-backup.timer` active and `gdrive-crypt:` successfully uploading encrypted backups to your 5 TB Google Drive (`tail /var/log/rclone-gdrive-backup.log` or `rclone ls gdrive:Homelab_Encrypted_Vault/`).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 6.2.3: Enable Tailscale Subnet Router (`homelab-pi`) & Expand Uptime Kuma to All 10 Pillars
1. **Remote Zero-Port Access to `UNAS 2` (`10.0.0.202`) & `USW-Enterprise-8-PoE` (`10.0.0.203`):**
   SSH into `homelab-pi` (`ssh homelab-pi`) and advertise strictly those two `/32` appliance host routes over Tailscale (so you can reach UniFi Drive and your switch remotely without opening a single router port):
   ```bash
   echo 'net.ipv4.ip_forward = 1' | sudo tee /etc/sysctl.d/99-tailscale.conf
   sudo sysctl -p /etc/sysctl.d/99-tailscale.conf
   sudo tailscale set --advertise-routes=10.0.0.202/32,10.0.0.203/32
   ```
   Open **Tailscale Admin Console (`https://login.tailscale.com/admin/machines`) $\rightarrow$ `homelab-pi` $\rightarrow$ Edit route settings** and approve both `/32` routes!

2. **Expand Uptime Kuma (`http://10.0.0.200:3001`) to Monitor All 10 Homelab Pillars:**
   In Uptime Kuma, add these **6 new Chapter 3 monitors** alongside your 4 existing Chapter 2 monitors:

| # | Monitor Friendly Name | Monitor Type | Target URL / IP | Notes |
| :-: | :--- | :--- | :--- | :--- |
| 1–4 | *(Existing Ch. 2 Monitors)* | Ping / HTTP / DNS | `homelab-gpu`, Immich `:2283`, CUDA ML `:3003`, Pi-hole `:53` | Already active |
| **5** | **Node 3: UniFi `UNAS 2` NAS** | `HTTP(s)` *(Ignore TLS error)* | `https://10.0.0.202` | UniFi OS & Drive appliance |
| **6** | **Core Switch: `USW-Enterprise-8-PoE`** | `Ping` | `10.0.0.203` | Sub-ms core fabric telemetry |
| **7** | **UniFi Network Server (`homelab-gpu`)** | `HTTP(s)` *(Ignore TLS error)* | `https://10.0.0.201:8443` | Self-hosted switch SDN controller |
| **8** | **FileBrowser Quantum (`homelab-gpu`)** | `HTTP(s)` | `http://10.0.0.201:8900` | Web file manager over `UNAS 2` |
| **9** | **Paperless-ngx OCR Vault (`homelab-gpu`)** | `HTTP(s)` | `http://10.0.0.201:8000` | Document & PDF/A archive |
| **10** | **Mail-Archiver (`homelab-gpu`)** | `HTTP(s)` | `http://10.0.0.201:5000` | `.edu` & Gmail searchable archive |

### 🖼️ Task 6.2.3 Verification: Complete 10-Monitor Homelab Observability Dashboard

**Verification Objective:** Capture a screenshot of **Uptime Kuma** (`http://10.0.0.200:3001`) showing `100%` green status across all 10 infrastructure, storage, document, email, and AI monitors.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Chapter 3 Final Checkpoint & Complete Personal Cloud Architecture Summary

Congratulations! You have built a genuinely enterprise-grade, zero-subscription-lock-in **3-Node Personal Cloud & Managed 2.5GbE Homelab** backed by true **3-2-1 zero-knowledge offsite backups**! Confirm every Chapter 3 milestone below:

- [ ] **Ubiquiti Hardware Integration:** `2× 4 TB UACC-HDD-S-4TB` installed powered-off in `UNAS 2` (RAID 1, ~3.64 TiB usable), powered via its UPS-backed **60W PoE++ injector** inline on `USW-Enterprise-8-PoE` Port 4, neatly cabled with `UACC-CABLE-PATCH-0.4M-BL-10` blue patch cords at **2.5 GbE**.
- [ ] **Managed Enterprise Fabric:** Self-hosted **UniFi Network Application** + pinned `mongo:7.0` running on `homelab-gpu:8443`, `USW-Enterprise-8-PoE` (`10.0.0.203`) adopted, ports labeled, and **PoE = Off** hygiene enforced on Ports 1–4.
- [ ] **UniFi Drive & Resilient NFS:** `HomelabBackups` & `MediaArchive` protected by scheduled Btrfs local snapshots and mounted on both Linux nodes via boot-safe `vers=3,nolock,_netdev,nofail,x-systemd.automount`.
- [ ] **University OneDrive Migration & File Management:** All graduating coursework/files rescued from university OneDrive onto `UNAS 2` RAID 1 and browsable anywhere via **FileBrowser Quantum (`homelab-gpu:8900`)**.
- [ ] **Document & Email Preservation:** **Paperless-ngx (`homelab-gpu:8000`)** providing automatic OCR + Office `.docx`/`.xlsx`/`.pptx` indexing over `UNAS 2`, and **Mail-Archiver (`homelab-gpu:5000`)** preserving `.edu` and personal Gmail correspondence with attachments.
- [ ] **True 3-2-1 Zero-Knowledge Cloud Backups & Observability:** Nightly `systemd` timers (`03:00 AM` Pi $\rightarrow$ `03:30 AM` GPU $\rightarrow$ `04:30 AM` `rclone crypt` to your included **5 TB Google Drive** with `--max-transfer 700G`), `rclone.conf` encryption keys saved offline, Tailscale `/32` subnet routes approved, and all **10 homelab pillars** green in **Uptime Kuma**!